# Create an editable cohort query

## Goal

Define a small query mapping, execute it, and reshape its body-weight and
hematocrit observations into an indexed daily table.

In [ ]:
import os
from pathlib import Path

source_from_environment = os.environ.get("VCGEN_SEND_ARCHIVE")
if source_from_environment:
    SEND_SOURCE = Path(source_from_environment)
else:
    SEND_SOURCE = next(
        (
            root / "rodent_db_export.zip"
            for root in (Path.cwd(), *Path.cwd().parents)
            if (root / "rodent_db_export.zip").is_file()
        ),
        Path("rodent_db_export.zip"),
    )

EXTRACTION_CACHE = Path(
    os.environ.get("VCGEN_QUERY_CACHE", SEND_SOURCE.parent / ".vcgen_query_cache")
)

if not SEND_SOURCE.exists():
    raise FileNotFoundError(
        "Set VCGEN_SEND_ARCHIVE to a SEND ZIP archive or extracted CSV directory; "
        f"got {SEND_SOURCE}"
    )

## 1. Define the query

In [ ]:
import vcgen_query as vq

query_mapping = {
    "contract": "vcgen.cohort_query.v4",
    "query_id": "example_rat_body_weight",
    "predicate": {
        "group": "and",
        "children": [
            {
                "field": {
                    "kind": "derived",
                    "name": "dose_duration_days",
                    "value_type": "number",
                },
                "operator": "between",
                "value": [28, 28],
            },
            {
                "field": {
                    "kind": "send_column",
                    "domain": "DM",
                    "column": "SPECIES",
                    "value_type": "string",
                },
                "operator": "eq",
                "value": "RAT",
            },
            {
                "field": {
                    "kind": "send_column",
                    "domain": "DM",
                    "column": "SEX",
                    "value_type": "string",
                },
                "operator": "in",
                "value": ["F", "M"],
            },
            {
                "field": {
                    "kind": "send_column",
                    "domain": "EX",
                    "column": "EXROUTE",
                    "value_type": "string",
                },
                "operator": "eq",
                "value": "ORAL GAVAGE",
            },
        ],
    },
    "observation_contract": {
        "endpoints": [
            {"name": "BW", "domain": "BW", "test_codes": ["BW"]},
            {
                "name": "LB_HCT",
                "domain": "LB",
                "test_codes": ["HCT"],
                "specimens": ["BLOOD", "WHOLE BLOOD"],
                "specimen_match": "exact",
                "source_units": ["%"],
                "target_unit": "%",
            },
        ]
    },
}

query = vq.parse_query(query_mapping)
{
    "query_id": query.query_id,
    "predicate_fingerprint": vq.fingerprint_predicate(query.predicate),
}

## 2. Execute and reshape

In [ ]:
result = vq.execute_query(
    SEND_SOURCE,
    query,
    extract_dir=EXTRACTION_CACHE,
)

{
    "subjects": len(result.subjects),
    "observations": len(result.observations),
}

In [ ]:
KEY_COLUMNS = ["study_id", "subject_id", "study_day"]
daily_values = (
    vq.to_daily_frame(result)
    .values.set_index(KEY_COLUMNS)
    .sort_index()
)

daily_values[["BW", "LB_HCT"]].dropna(how="all").head(12)

## Takeaway

Edit the predicate to change cohort membership. Edit the observation contract
to change extracted endpoints. Reshape the returned DataFrames without changing
either query component.